In [2]:
%pip install opendatasets numpy pandas matplotlib seaborn streamlit

Note: you may need to restart the kernel to use updated packages.


In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import opendatasets as od

In [4]:
%pip install numpy pandas matplotlib seaborn opendatasets

Note: you may need to restart the kernel to use updated packages.


In [5]:
od.download("https://www.kaggle.com/datasets/prokshitha/home-value-insights/data")

Skipping, found downloaded files in ".\home-value-insights" (use force=True to force download)


In [6]:
df = pd.read_csv(r"C:\Users\zakar\Desktop\Linear regression\home-value-insights\house_price_regression_dataset.csv")

In [7]:
df

,Square_Footage,Num_Bedrooms,Num_Bathrooms,Year_Built,Lot_Size,Garage_Size,Neighborhood_Quality,House_Price
0,1360,2,1,1981,0.599637,0,5,2.623829e+05
1,4272,3,3,2016,4.753014,1,6,9.852609e+05
2,3592,1,2,2016,3.634823,0,9,7.779774e+05
3,966,1,2,1977,2.730667,1,8,2.296989e+05
4,4926,2,1,1993,4.699073,0,8,1.041741e+06
...,...,...,...,...,...,...,...,...
995,3261,4,1,1978,2.165110,2,10,7.014940e+05
996,3179,1,2,1999,2.977123,1,10,6.837232e+05
997,2606,4,2,1962,4.055067,0,2,5.720240e+05
998,4723,5,2,1950,1.930921,0,7,9.648653e+05


In [8]:
def linear_regression(x,w,b):
    linear_regression_model = np.dot(x,w) + b
    return linear_regression_model

In [9]:
def mean_squared_error(x, y,w,b):
    m = len(y)
    linear_regression_model = linear_regression(x,w,b)
    mse = (1/(2*m)) * np.sum(np.square(y - linear_regression_model))
    return mse

In [10]:
def gradient_descent(x,y,w,b,a):
    the_dervative_w =  -x * 2 * (y - linear_regression(x,w,b))
    the_dervative_b = -2 * (y - linear_regression(x,w,b))
    w = w - a * the_dervative_w
    b = b - a * the_dervative_b
    return w, b

In [11]:
# --- Feature Scaling Implementation ---
def rescale_features(X):
    """
    Applies min-max scaling to ensure all features are in a 0 to 1 range.
    This reshapes the [[Cost Function]] contours from ellipses to circles.
    """
    # Calculate the maximum and minimum values for each column (feature)
    X_max = np.max(X, axis=0)
    X_min = np.min(X, axis=0)

    # Broadcast element-wise operations to rescale
    X_rescaled = (X - X_min) / (X_max - X_min)
    return X_rescaled

In [12]:
# --- Feature Engineering Implementation ---
M = df["Year_Built"]
for i in range(len(M)):
    M[i] = 2023 - M[i]
    df["Year_Built"] = M

In [13]:
df.describe()

,Square_Footage,Num_Bedrooms,Num_Bathrooms,Year_Built,Lot_Size,Garage_Size,Neighborhood_Quality,House_Price
count,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1.000000e+03
mean,2815.422000,2.990000,1.973000,36.450000,2.778087,1.022000,5.615000,6.188610e+05
std,1255.514921,1.427564,0.820332,20.632916,1.297903,0.814973,2.887059,2.535681e+05
min,503.000000,1.000000,1.000000,1.000000,0.506058,0.000000,1.000000,1.116269e+05
25%,1749.500000,2.000000,1.000000,18.750000,1.665946,0.000000,3.000000,4.016482e+05
50%,2862.500000,3.000000,2.000000,37.000000,2.809740,1.000000,6.000000,6.282673e+05
75%,3849.500000,4.000000,3.000000,54.000000,3.923317,2.000000,8.000000,8.271413e+05
max,4999.000000,5.000000,3.000000,73.000000,4.989303,2.000000,10.000000,1.108237e+06


In [14]:
#--- Remove Outliers Implementation ---
def remove_outliers(df: pd.DataFrame, multiplier: float = 1.5) -> pd.DataFrame:
    """Removes entire rows containing outliers in any numeric column."""
    num_cols = df.select_dtypes(include=[np.number]).columns
    if len(num_cols) == 0:
        return df.copy()

    Q1 = df[num_cols].quantile(0.25)
    Q3 = df[num_cols].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - multiplier * IQR
    upper_bound = Q3 + multiplier * IQR

    outlier_mask = (df[num_cols] < lower_bound) | (df[num_cols] > upper_bound)
    return df[~outlier_mask.any(axis=1)]

In [15]:
df = remove_outliers(df)

In [16]:
df.describe()

,Square_Footage,Num_Bedrooms,Num_Bathrooms,Year_Built,Lot_Size,Garage_Size,Neighborhood_Quality,House_Price
count,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1.000000e+03
mean,2815.422000,2.990000,1.973000,36.450000,2.778087,1.022000,5.615000,6.188610e+05
std,1255.514921,1.427564,0.820332,20.632916,1.297903,0.814973,2.887059,2.535681e+05
min,503.000000,1.000000,1.000000,1.000000,0.506058,0.000000,1.000000,1.116269e+05
25%,1749.500000,2.000000,1.000000,18.750000,1.665946,0.000000,3.000000,4.016482e+05
50%,2862.500000,3.000000,2.000000,37.000000,2.809740,1.000000,6.000000,6.282673e+05
75%,3849.500000,4.000000,3.000000,54.000000,3.923317,2.000000,8.000000,8.271413e+05
max,4999.000000,5.000000,3.000000,73.000000,4.989303,2.000000,10.000000,1.108237e+06


In [17]:
df = rescale_features(df)

In [18]:
df

,Square_Footage,Num_Bedrooms,Num_Bathrooms,Year_Built,Lot_Size,Garage_Size,Neighborhood_Quality,House_Price
0,0.190614,0.25,0.0,0.569444,0.020873,0.0,0.444444,0.151269
1,0.838301,0.50,1.0,0.083333,0.947295,0.5,0.555556,0.876606
2,0.687055,0.00,0.5,0.083333,0.697880,0.0,0.888889,0.668617
3,0.102980,0.00,0.5,0.625000,0.496205,0.5,0.777778,0.118474
4,0.983763,0.25,0.0,0.402778,0.935263,0.0,0.777778,0.933278
...,...,...,...,...,...,...,...,...
995,0.613434,0.75,0.0,0.611111,0.370056,1.0,1.000000,0.591874
996,0.595196,0.00,0.5,0.319444,0.551178,0.5,1.000000,0.574042
997,0.467749,0.75,0.5,0.833333,0.791616,0.0,0.111111,0.461963
998,0.938612,1.00,0.5,1.000000,0.317820,0.0,0.666667,0.856141


In [19]:
#--- spliting the data into features and target variable ---
x = df.drop("House_Price", axis=1)
y = df["House_Price"]

In [20]:
#--- selecting a randome seed(state) for reproducibility ---
np.random.seed(42)

In [21]:
m = x.shape[0]
test_size = 0.2 
split_index = int(m*(1-test_size))

In [22]:
x

,Square_Footage,Num_Bedrooms,Num_Bathrooms,Year_Built,Lot_Size,Garage_Size,Neighborhood_Quality
0,0.190614,0.25,0.0,0.569444,0.020873,0.0,0.444444
1,0.838301,0.50,1.0,0.083333,0.947295,0.5,0.555556
2,0.687055,0.00,0.5,0.083333,0.697880,0.0,0.888889
3,0.102980,0.00,0.5,0.625000,0.496205,0.5,0.777778
4,0.983763,0.25,0.0,0.402778,0.935263,0.0,0.777778
...,...,...,...,...,...,...,...
995,0.613434,0.75,0.0,0.611111,0.370056,1.0,1.000000
996,0.595196,0.00,0.5,0.319444,0.551178,0.5,1.000000
997,0.467749,0.75,0.5,0.833333,0.791616,0.0,0.111111
998,0.938612,1.00,0.5,1.000000,0.317820,0.0,0.666667


In [23]:
def gradient_descent_vectorized(x,y,w,b,a):
    m = len(y)
    y_hat_predict = linear_regression(x,w,b)
    dw = 2 * np.dot(x.T, y_hat_predict - y) / m
    db = 2 * np.sum(y_hat_predict - y) / m  # Fix: remove the negative
    w = w - a*dw
    b = b - a*db
    return w,b

In [24]:
#---Shuffling the data---
indices = np.arange(m)
np.random.shuffle(indices)
x_shuffled = x.iloc[indices]
y_shuffled = y.iloc[indices]

In [25]:
#---spliting the data into train and test---
x_train = x_shuffled[:split_index] #taking hard copies and spliting
y_train = y_shuffled[:split_index]
x_test = x_shuffled[split_index:]
y_test = y_shuffled[split_index:]

In [26]:
w = np.zeros(x_train.shape[1])
b = 0
learning_rate = 0.01
epochs = 1000

In [27]:
# Train
for epoch in range(epochs):
    w, b = gradient_descent_vectorized(x_train.values, y_train.values, w, b, learning_rate)
    if epoch % 100 == 0:
        loss = mean_squared_error(x_train.values, y_train.values, w, b)
        print(f"Epoch {epoch}, Loss: {loss}")

Epoch 0, Loss: 0.14582798604280384
Epoch 100, Loss: 0.020196677260293817
Epoch 200, Loss: 0.014593848129974064
Epoch 300, Loss: 0.010654228532898111
Epoch 400, Loss: 0.007850147088273805
Epoch 500, Loss: 0.00583353792799269
Epoch 600, Loss: 0.004370170933929551
Epoch 700, Loss: 0.00329982935371407
Epoch 800, Loss: 0.0025113534338657055
Epoch 900, Loss: 0.0019266749404589422


In [28]:
# Make predictions
y_pred_train = linear_regression(x_train.values, w, b)
y_pred_test = linear_regression(x_test.values, w, b)

# Mean Squared Error
def mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

# R-squared
def r_squared(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1 - (ss_res / ss_tot)

# Mean Absolute Error
def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

# Evaluate
print(f"Train MSE: {mse(y_train, y_pred_train)}")
print(f"Test MSE: {mse(y_test, y_pred_test)}")
print(f"Train R²: {r_squared(y_train, y_pred_train)}")
print(f"Test R²: {r_squared(y_test, y_pred_test)}")
print(f"Test MAE: {mae(y_test, y_pred_test)}")

Train MSE: 0.0029883326962055158
Test MSE: 0.003160680444904333
Train R²: 0.953127362478099
Test R²: 0.9537134499018836
Test MAE: 0.04797242631711634


In [31]:
%%writefile app.py
import streamlit as st
import numpy as np
import json
import os

@st.cache_data
def load_model():
    if not os.path.exists("model_artifacts.json"):
        st.error("Model artifacts not found. Please run the export script first.")
        st.stop()
    with open("model_artifacts.json", "r") as f:
        return json.load(f)

artifacts = load_model()
w = np.array(artifacts["weights"]).reshape(-1, 1)
b = artifacts["bias"]
x_min = np.array(artifacts["x_min"])
x_max = np.array(artifacts["x_max"])
y_min = artifacts["y_min"]
y_max = artifacts["y_max"]

st.title("Real Estate Price Predictor")
st.markdown("Enter property details to estimate the market value.")

col1, col2 = st.columns(2)

with col1:
    sqft = st.number_input("Square Footage", min_value=500, max_value=10000, value=2500)
    bedrooms = st.number_input("Number of Bedrooms", min_value=1, max_value=10, value=3)
    bathrooms = st.number_input("Number of Bathrooms", min_value=1.0, max_value=5.0, value=2.0, step=0.5)
    year_built = st.number_input("Year Built", min_value=1800, max_value=2024, value=2000)

with col2:
    lot_size = st.number_input("Lot Size (Acres)", min_value=0.1, max_value=10.0, value=1.0, step=0.1)
    garage_size = st.number_input("Garage Size (Cars)", min_value=0, max_value=5, value=2)
    neighborhood_quality = st.slider("Neighborhood Quality (1-10)", min_value=1, max_value=10, value=5)

if st.button("Predict Price", type="primary"):
    engineered_year = 2023 - year_built
    raw_features = np.array([sqft, bedrooms, bathrooms, engineered_year, lot_size, garage_size, neighborhood_quality])
    scale_range = np.where((x_max - x_min) == 0, 1e-8, (x_max - x_min))
    scaled_features = (raw_features - x_min) / scale_range
    y_scaled_pred = np.dot(scaled_features, w) + b
    predicted_price = y_scaled_pred[0] * (y_max - y_min) + y_min
    st.success(f"Estimated Property Value: ${predicted_price:,.2f}")

Writing app.py


In [32]:
import json

# Export model artifacts
x_min_list = x.min().tolist()
x_max_list = x.max().tolist()
y_min = y.min()
y_max = y.max()

artifacts = {
    "weights": w.tolist(),
    "bias": float(b),
    "x_min": x_min_list,
    "x_max": x_max_list,
    "y_min": float(y_min),
    "y_max": float(y_max)
}

with open("model_artifacts.json", "w") as f:
    json.dump(artifacts, f, indent=2)

print("Model artifacts exported to model_artifacts.json")

Model artifacts exported to model_artifacts.json
